# The Economic Cost of Grid Congestion

When two neighbouring bidding zones have different day-ahead prices, power is supposed to flow from the cheap zone to the expensive one until prices converge. Sometimes it can't &mdash; the interconnector between them is already running at its physical capacity limit (its net transfer capacity, or NTC). Voltpath calls this a **congestion** flow: the price spread persists because the grid, not the market, is the binding constraint.

Every hour a border sits congested, that persistent price spread is a real economic cost: consumers on the expensive side pay more than they would if the interconnector had spare capacity, and the potential gains from trade go unrealised. This notebook estimates that cost directly from Voltpath data: for every congested observation, `|price difference| x |flow| x duration` gives the implied cost in EUR, and summing that across borders and time produces a leaderboard of Europe's most expensive bottlenecks.

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai). Physical flow data is CC-BY 4.0; day-ahead prices are reused under ENTSO-E's general Transparency Platform terms.

**Before you run this:** sign up at [voltpath.ai](https://voltpath.ai) for a free personal API key, then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY` (or leave `.env` empty and you'll be prompted).

In [ ]:
import getpass
import os

import pandas as pd
import plotly.express as px
import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = "https://api.voltpath.ai"
API_KEY = os.environ.get("VOLTPATH_API_KEY") or getpass.getpass("Voltpath API key (from voltpath.ai): ")
HEADERS = {"x-api-key": API_KEY} if API_KEY else {}

## Pull flow data

`GET /api/v1/flows` already joins physical flow with day-ahead prices on both sides of the border and classifies each hourly (or sub-hourly) observation into `Convergence`, `Congestion`, `Counter-intuitive`, or `Outage` &mdash; see [Flow Types](https://docs.voltpath.ai/concepts/flow-types) for the full definitions. No client-side join needed.

We pull the last 30 days across every border at native resolution.

In [ ]:
resp = requests.get(
    f"{BASE_URL}/api/v1/flows",
    headers=HEADERS,
    params={"datetime_from": "now-30d", "datetime_to": "now", "resolution": "native"},
    timeout=60,
)
resp.raise_for_status()
payload = resp.json()
print(payload["meta"])

df = pd.DataFrame(payload["data"])
df["datetime"] = pd.to_datetime(df["datetime"])
df["border"] = df["bidding_zone_from"] + " \u2192 " + df["bidding_zone_to"]
print(f"{len(df):,} observations across {df['border'].nunique()} borders")
df.head()

## Compute the cost of congestion

Each row is one observation period; how long that period lasts depends on the border (some zones report at 15-minute resolution, others hourly). We infer each border's cadence from the gap between consecutive timestamps rather than assuming a fixed duration.

Congestion cost per observation is then `|price_delta| (EUR/MWh) x |net_flow_mw| (MW) x duration (h)`, in EUR.

In [ ]:
diffs_h = df.sort_values("datetime").groupby("border")["datetime"].diff().dt.total_seconds() / 3600
median_cadence = diffs_h.groupby(df["border"]).transform("median")
df["duration_h"] = diffs_h.fillna(median_cadence)

congestion = df[df["flow_type"] == "Congestion"].copy()
congestion["cost_eur"] = (
    congestion["price_delta"].abs() * congestion["net_flow_mw"].abs() * congestion["duration_h"]
)

total_cost = congestion["cost_eur"].sum()
print(f"{len(congestion):,} congested observations, implied cost: EUR {total_cost:,.0f}")

## The leaderboard: most expensive borders

A handful of interconnectors typically account for the bulk of the implied cost &mdash; usually where NTC is small relative to demand, or where one side has extreme price volatility.

In [ ]:
leaderboard = (
    congestion.groupby("border")["cost_eur"].sum().sort_values(ascending=False).head(15).reset_index()
)

fig = px.bar(
    leaderboard,
    x="cost_eur",
    y="border",
    orientation="h",
    title="Top 15 borders by implied congestion cost (last 30 days)",
    labels={"cost_eur": "Implied cost (EUR)", "border": "Border"},
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()

A few outliers are worth reading in context rather than taking at face value: borders touching Ukraine (`UA_IPS`) show extreme implied costs, driven by wartime grid instability producing very large and volatile price spreads rather than routine market congestion. Worth excluding or footnoting depending on what the leaderboard shows when you run it.

## Congestion cost over time

Daily totals across all borders show whether congestion cost is trending, seasonal, or driven by a handful of spike days.

In [ ]:
daily = congestion.set_index("datetime")["cost_eur"].resample("1D").sum().reset_index()

fig = px.line(
    daily,
    x="datetime",
    y="cost_eur",
    title="Daily implied congestion cost, all borders",
    labels={"datetime": "Day", "cost_eur": "Implied cost (EUR)"},
)
fig.show()

## Takeaways

- Congestion cost concentrates heavily: a small number of borders usually account for most of the total.
- The metric here is an *implied* cost &mdash; the value that would be unlocked by removing the physical bottleneck, not a cash flow anyone actually pays. It's the standard way analysts size the benefit case for new interconnector or grid-reinforcement investment.
- Try narrowing to a single border with `border="ZONE_FROM->ZONE_TO"` on the `/flows` call, or widening the window past 30 days, to see how a specific bottleneck evolves.

---

Data: [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai).